# Evaluate a dataset-level mHSC GRN with scCAFM SFM

This tutorial evaluates a mean dataset-level SFM GRN for `mHSC-E.h5ad` against `mHSC-ChIP-seq.csv`. The workflow keeps a few medium-sized objects visible so it is easy to understand what is being evaluated: the reference GRN table, the preprocessed AnnData, the dense forward GRN tensor, the mean dataset-level GRN shape, and the final metrics table.

Unlike the pancreas tutorial, this notebook is focused on dataset-level evaluation instead of cell-specific edge export.


## 1. Environment

Run this notebook from the scCAFM repository root, or from `docs/`. The first cell resolves the repository root and adds it to `sys.path` so the local `src` package can be imported directly.


In [8]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
repo_root = start if (start / "src").exists() else start.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the scCAFM repository root or docs/.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from src.inference import grn

print(f"Repository root: {repo_root}")


Repository root: /data1021/xukaichen/scCAFM


## 2. Configure inputs

The mHSC dataset and ChIP-seq network paths are the runnable defaults for this tutorial. `n_top_genes` is the HVG target-gene budget for GRN-inference preprocessing. Preserved TFs are added before those target genes, so `max_length` must leave room for both.

`candidate_source_universe="supported_tfs"` means evaluation source candidates are restricted to TFs supported by the mapped reference/assets. Targets are the active genes in the processed dataset.


In [9]:
input_h5ad = Path("/data1021/xukaichen/data/GRN/datasets/mHSC-E.h5ad")
reference_grn_csv = Path("/data1021/xukaichen/data/GRN/networks/mHSC-ChIP-seq.csv")
output_metrics_csv = repo_root / "results" / "mhsc_e_dataset_grn_metrics.csv"

model_source = repo_root / "assets"
checkpoint_path = None

batch_size = 64
max_length = 4096
n_top_genes = 1000
metric_names = ("auprc", "auroc", "ep")
candidate_source_universe = "supported_tfs"
predicted_grn_head_n = 10

gene_key = None
species_key = "species"
disease_key = "disease"

print(f"Input: {input_h5ad}")
print(f"Reference GRN: {reference_grn_csv}")
print(f"Output: {output_metrics_csv}")


Input: /data1021/xukaichen/data/GRN/datasets/mHSC-E.h5ad
Reference GRN: /data1021/xukaichen/data/GRN/networks/mHSC-ChIP-seq.csv
Output: /data1021/xukaichen/scCAFM/results/mhsc_e_dataset_grn_metrics.csv


## 3. Inspect reference GRN

Before running evaluation, inspect the ChIP-seq reference network. The evaluator maps `Gene1` and `Gene2` to model-supported genes, then uses the mapped pairs as positive TF-target edges.


In [10]:
reference_grn_df = pd.read_csv(reference_grn_csv)

print(f"Reference edges: {len(reference_grn_df):,}")
print(f"Unique source TFs: {reference_grn_df['Gene1'].nunique():,}")
print(f"Unique target genes: {reference_grn_df['Gene2'].nunique():,}")
reference_grn_df.head()


Reference edges: 1,078,888
Unique source TFs: 137
Unique target genes: 19,323


,Gene1,Gene2,Score
0,AICDA,SSPN,1625.0
1,AICDA,MRPL44,1946.0
2,AICDA,EID1,2109.0
3,AICDA,PIP4P1,1841.0
4,AICDA,FAM149B,1531.0


## 4. Preprocess AnnData

This tutorial uses GRN-inference HVG preprocessing: mouse asset-supported TFs are preserved first, then HVG target genes are selected from the remaining genes. This differs from the pancreas tutorial, where the active gene universe is selected by normal HVG preprocessing without forcing TF preservation.

The processed AnnData defines the fixed gene/token universe used for all cells in `forward_results` and for dataset-level evaluation.


In [11]:
run = grn.prepare(
    input_h5ad=input_h5ad,
    mode=grn.PreprocessMode.TF_PRESERVED_HVG,
    model_source=model_source,
    checkpoint_path=checkpoint_path,
    config_path=repo_root / "configs" / "eval_grn.yaml",
    batch_size=batch_size,
    max_length=max_length,
    n_top_genes=n_top_genes,
    preserve_tf_species="mouse",
    gene_key=gene_key,
    species_key=species_key,
    disease_key=disease_key,
)

processed_adata = run.adata
print(f"Raw shape: {run.raw_adata.shape}")
print(f"Processed shape: {processed_adata.shape}")
print(f"Active gene count: {processed_adata.n_vars:,}")
processed_adata


Raw shape: (1071, 4762)
Processed shape: (1071, 1321)
Active gene count: 1,321


AnnData object with n_obs × n_vars = 1071 × 1321
    obs: 'species', 'disease', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'preserved_gene', 'hvg_target_candidate', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'hvg_target_gene'

## 5. Collect forward results

`forward_results` is a dense NumPy array with shape `(cell, gene, gene)` after padding gene slots are removed. The second and third axes use the same active gene order, so each cell has one predicted GRN matrix over the processed gene universe.


In [12]:
forward_results = grn.predict(run)

print(f"forward_results shape: {forward_results.shape}")
print(f"forward_results dtype: {forward_results.dtype}")


forward_results shape: (1071, 1321, 1321)
forward_results dtype: float32


## 6. Aggregate dataset-level GRN

Dataset-level evaluation uses the mean GRN across all processed cells. This matrix is useful for evaluation, but it is not saved or plotted here because it is still a large gene-by-gene matrix.


In [13]:
dataset_grn = forward_results.mean(axis=0)

print(f"dataset_grn shape: {dataset_grn.shape}")


dataset_grn shape: (1321, 1321)


## 7. Preview predicted GRN

This preview converts the highest-scoring entries in the mean dataset-level GRN into a small table, similar to `reference_grn_df.head()`. It is only for inspection: the full dense matrix is not saved, and the metrics are still computed by `grn.evaluate(...)` from `forward_results`.


In [14]:
token_template = run.forward_token_template
if token_template is None:
    raise RuntimeError("Run grn.predict(run) before previewing predicted GRNs.")

input_ids = token_template["input_ids"][0].detach().cpu().long().numpy()
non_tf_mask = token_template["non_tf_mask"][0].detach().cpu().bool().numpy()

token_to_gene = {}
for _, row in run.data_assets.token_dict.iterrows():
    token_id = int(row["token_index"])
    gene_symbol = row.get("gene_symbol")
    gene_id = row.get("gene_id")
    if pd.notna(gene_symbol) and str(gene_symbol).strip():
        token_to_gene[token_id] = str(gene_symbol)
    elif pd.notna(gene_id) and str(gene_id).strip():
        token_to_gene[token_id] = str(gene_id)
    else:
        token_to_gene[token_id] = str(token_id)

source_positions = np.flatnonzero(~non_tf_mask)
target_positions = np.arange(dataset_grn.shape[1])
candidate_scores = dataset_grn[np.ix_(source_positions, target_positions)].copy()
self_loop_mask = input_ids[source_positions, None] == input_ids[target_positions][None, :]
candidate_scores[self_loop_mask] = -np.inf

flat_order = np.argsort(candidate_scores.ravel())[::-1][:predicted_grn_head_n]
source_rank, target_rank = np.unravel_index(flat_order, candidate_scores.shape)

predicted_grn_df = pd.DataFrame(
    {
        "Gene1": [token_to_gene[int(input_ids[source_positions[i]])] for i in source_rank],
        "Gene2": [token_to_gene[int(input_ids[target_positions[j]])] for j in target_rank],
        "score": candidate_scores[source_rank, target_rank],
    }
)

predicted_grn_df.head(predicted_grn_head_n)


,Gene1,Gene2,score
0,SNRPB,EBF1,0.600641
1,NFE2,EBF1,0.599731
2,SNRPB,MYC,0.598161
3,SNRPB,ZEB2,0.596985
4,NFE2,MYC,0.596821
5,HNRNPAB,EBF1,0.595804
6,SNRPB,MMS19,0.595792
7,NFE2,ZEB2,0.595687
8,YBX1,EBF1,0.595260
9,SNRPB,NFKB1,0.594802


## 8. Evaluate dataset-level GRN

The evaluator maps the ChIP-seq reference to the active model genes, builds candidate TF-to-active-gene pairs, and compares the mean predicted GRN against the mapped positives.

Here we report AUPRC, AUROC, and early precision to keep the evaluation output focused.


In [15]:
metrics_df = grn.evaluate(
    run,
    forward_results,
    reference_grn_csv=reference_grn_csv,
    output_metrics_csv=output_metrics_csv,
    candidate_source_universe=candidate_source_universe,
    metric_names=metric_names,
    dataset_name="mHSC-E",
    dataset_path=input_h5ad,
    reference_grn_name="mHSC-ChIP-seq",
)

print(f"Saved metrics to {output_metrics_csv}")
metrics_df


Saved metrics to /data1021/xukaichen/scCAFM/results/mhsc_e_dataset_grn_metrics.csv


,aggregation,dataset_name,dataset_path,evaluation_grn_name,evaluation_grn_path,num_cells,num_candidate_edges,num_positive_edges,auprc,auroc,ep
0,dataset,mHSC-E,/data1021/xukaichen/data/GRN/datasets/mHSC-E.h5ad,mHSC-ChIP-seq,/data1021/xukaichen/data/GRN/networks/mHSC-ChI...,1071,54120,26028,0.621298,0.642611,0.588251


## 9. Interpret metrics

- `auprc`: precision-recall quality over candidate edges.
- `auroc`: overall ranking quality across positive and negative candidate edges.
- `ep`: early precision among the top-ranked candidate edges.

The count columns are also useful sanity checks: `num_candidate_edges` is the number of evaluated candidate pairs, and `num_positive_edges` is the number of mapped ChIP-seq positives among them.


## 10. Notes

- Preserved TFs are loaded from the mouse asset TF table before HVG target-gene selection.
- With `candidate_source_universe="supported_tfs"`, source candidates are mapped reference `Gene1` TFs.
- Targets are active processed genes in the fixed dataset-level token template.
- `dataset_grn` is computed only for inspection here; `grn.evaluate(...)` performs the same mean aggregation internally from `forward_results`.
- `predicted_grn_df` is a compact preview of top-ranked dataset-level predictions and is not used as the saved metrics output.
